In [17]:
import os, platform
print("on Colab:", os.path.exists("/content"), "| OS:", platform.system())
!nvidia-smi -L

on Colab: True | OS: Linux
GPU 0: Tesla T4 (UUID: GPU-6ef7e9b4-578e-1d28-583e-caa263be677f)


# A1: Tokens, tensors, shapes, logits

**Goal:** follow one prompt through TinyLlama and understand every shape.

```
text --tokenizer--> ids (N,) --embedding--> (N, 2048) --22 decoder layers--> (N, 2048) --lm_head--> logits (N, 32000)
```

Key ideas:
1. The model never sees text, only integer token ids.
2. We keep ids as one flat 1-D tensor `(N,)`. This is the *flattened-token* convention the whole engine will use.
3. Row `i` of the logits scores the token that comes **after** position `i`. Generation only needs the last row.
4. Position `i` cannot see later tokens (causality). This is the reason a KV cache is valid (Part C).

In [18]:
# Setup: clone or pull the repo, cd into it, put it on sys.path, install pytest.
import os, subprocess, sys

REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    # Colab may hold untracked files (e.g. results written by a notebook) that clash with a pull.
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

2a5fe3e Notebooks: setup cell fetches and hard-resets instead of pull



In [19]:
import importlib
import tinyserve.basics
importlib.reload(tinyserve.basics)  # picks up repo edits after a git pull without restarting the kernel
from tinyserve.envinfo import collect_env

collect_env()

{'python': '3.13.16',
 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39',
 'git_sha': '2a5fe3e',
 'gpu': 'Tesla T4',
 'torch': '2.11.0+cu130',
 'cuda': '13.0',
 'transformers': '5.20.2'}

## 1. Load the model
`load_model` returns the tokenizer and the Hugging Face model in FP16 on the GPU. For now the HF model is our *reference*; from Part B we build our own and compare against it.

In [20]:
import torch
from tinyserve.basics import load_model, encode, forward_logits, next_token_greedy, shape_report

DEVICE = "cuda"
tokenizer, model = load_model(device=DEVICE, dtype=torch.float16)
print(model.config)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "float16",
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 5632,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 22,
  "num_key_value_heads": 4,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.20.2",
  "use_cache": true,
  "vocab_size": 32000
}



## 2. Text to tokens
Notice the first id is BOS (`<s>`, id 1), added by the tokenizer. Words may split into pieces, and `▁` marks a leading space.

In [21]:
PROMPT = "The capital of France is"
prompt_ids = encode(tokenizer, PROMPT, DEVICE)
print("ids:", prompt_ids.tolist())
print("tokens:", tokenizer.convert_ids_to_tokens(prompt_ids.tolist()))
print("shape:", tuple(prompt_ids.shape), "dtype:", prompt_ids.dtype, "device:", prompt_ids.device)

ids: [1, 450, 7483, 310, 3444, 338]
tokens: ['<s>', '▁The', '▁capital', '▁of', '▁France', '▁is']
shape: (6,) dtype: torch.int64 device: cuda:0


## 3. Shapes and sizes
- embedding output: `(N, 2048)`; logits: `(N, 32000)`.
- Weights: about 1.1B params x 2 bytes (FP16) = about 2.2 GB. Every decoded token must read all of them (Lesson A2).
- Logits for N tokens cost `N * 32000 * 2` bytes. For long prompts we will only compute the last row (Part D).

In [22]:
report = shape_report(tokenizer, model, PROMPT)
for key, value in report.items():
    print(f"{key:>18}: {value}")
print(f"weights: {report['weight_bytes'] / 1e9:.2f} GB")

              text: The capital of France is
            tokens: ['<s>', '▁The', '▁capital', '▁of', '▁France', '▁is']
   input_ids_shape: (6,)
   embedding_shape: (6, 2048)
      logits_shape: (6, 32000)
      logits_dtype: torch.float16
      logits_bytes: 384000
        num_params: 1100048384
      weight_bytes: 2200096768
weights: 2.20 GB


## 4. Logits: what the numbers mean
Logits are unnormalised scores over the 32000-token vocabulary. `softmax` turns them into probabilities. The **last row** predicts the next token.

In [23]:
logits = forward_logits(model, prompt_ids)
print("logits:", tuple(logits.shape), logits.dtype)

def show_top_k(logits_row, k):
    probs = torch.softmax(logits_row.float(), dim=-1)
    top = probs.topk(k)
    for prob, token_id in zip(top.values.tolist(), top.indices.tolist()):
        print(f"{prob:7.3%}  id={token_id:<6} {tokenizer.convert_ids_to_tokens(token_id)!r}")

show_top_k(logits[-1], 5)
print("greedy next token:", tokenizer.convert_ids_to_tokens(next_token_greedy(logits)))

logits: (6, 32000) torch.float16
50.947%  id=3681   '▁Paris'
16.540%  id=5982   '▁located'
 2.170%  id=29901  ':'
 2.170%  id=278    '▁the'
 2.103%  id=29973  '?'
greedy next token: ▁Paris


## 5. A first (slow) generation loop
Append the greedy token, run the model again on the **whole** sequence, repeat. This works but redoes all earlier work every step. Part C fixes it with a KV cache.

In [24]:
from tinyserve.basics import generate_greedy_naive

generated = generate_greedy_naive(model, prompt_ids, max_new_tokens=20)
print(tokenizer.decode(generated, skip_special_tokens=True))

The capital of France is Paris.

2. B. The capital of Germany is Berlin.

3. C


## 6. Causality
Append extra tokens and compare the logits of the original positions. They should not change. In FP16 on GPU the kernels for different lengths may round slightly differently, so we expect a tiny difference, not bit-equality. The FP32 CPU test in `tests/` checks this tightly.

In [25]:
from tinyserve.basics import prefix_logit_diff

extra = encode(tokenizer, " Paris and the weather is nice", DEVICE)[1:]  # drop the BOS
print("max |diff| on shared prefix (FP16):", prefix_logit_diff(model, prompt_ids, extra))

max |diff| on shared prefix (FP16): 0.0693359375


## 7. Run the lesson tests
Includes the `slow` test that checks the real TinyLlama config (22 layers, hidden 2048, 32 query heads, 4 KV heads, head_dim 64).

In [26]:
result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_a1_basics.py", "-v"], capture_output=True, text=True)
print(result.stdout[-3000:], result.stderr[-1000:])

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/tinyserve
configfile: pyproject.toml
plugins: typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.11, platformdirs-4.13.5
collecting ... collected 7 items

tests/test_a1_basics.py::test_logits_shape_is_one_row_per_token PASSED   [ 14%]
tests/test_a1_basics.py::test_greedy_uses_last_row PASSED                [ 28%]
tests/test_a1_basics.py::test_causality_later_tokens_do_not_change_earlier_logits PASSED [ 42%]
tests/test_a1_basics.py::test_prefix_logit_diff_is_tiny PASSED           [ 57%]
tests/test_a1_basics.py::test_naive_generate_appends_greedy_tokens PASSED [ 71%]
tests/test_a1_basics.py::test_changing_an_earlier_token_changes_later_logits PASSED [ 85%]
tests/test_a1_basics.py::test_real_tinyllama_config_and_shapes PASSED    [100%]

============================== 7 passed in 14.31s =======

In [27]:
import json
from tinyserve.envinfo import save_result

saved_path = save_result("A1_shapes", report)
print(saved_path)
print(saved_path.read_text())

/content/tinyserve/results/A1_shapes.json
{
  "name": "A1_shapes",
  "timestamp_utc": "2026-10-25T06:29:54Z",
  "env": {
    "python": "3.13.16",
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
    "git_sha": "2a5fe3e",
    "gpu": "Tesla T4",
    "torch": "2.11.0+cu130",
    "cuda": "13.0",
    "transformers": "5.20.2"
  },
  "data": {
    "text": "The capital of France is",
    "tokens": [
      "<s>",
      "\u2581The",
      "\u2581capital",
      "\u2581of",
      "\u2581France",
      "\u2581is"
    ],
    "input_ids_shape": [
      6
    ],
    "embedding_shape": [
      6,
      2048
    ],
    "logits_shape": [
      6,
      32000
    ],
    "logits_dtype": "torch.float16",
    "logits_bytes": 384000,
    "num_params": 1100048384,
    "weight_bytes": 2200096768
  }
}
